In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import math
from scipy.stats import norm # Perfect for cumulative normal distribution (N(d1), N(d2))
import yfinance as yf
import numpy as np
import pandas as pd
import random
from src.data_provider import *
from src.models.european import EuropeanOption
from src.models.american import AmericanOption
from src.base_option import FinancialOption


In [3]:
test_option = FinancialOption(ticker = "AAPL", strike = "random", expiry_date = "2026-09-18", option_type = "call")
test_option

FinancialOption(ticker='AAPL', strike='340.0', expiry_date='2026-09-18', spot_date='2026-07-13', option_type='call', underlying_spot='317.31', volatility='0.2704', risk_free_rate='0.0373', )

In [8]:
strike = 330

In [9]:
test_option = AmericanOption(ticker = "AAPL", strike = strike, expiry_date = "2026-09-18", option_type = "call")
test_option

AmericanOption(ticker='AAPL', strike='330.0', expiry_date='2026-09-18', spot_date='2026-07-13', option_type='call', underlying_spot='317.31', volatility='0.2778', risk_free_rate='0.0373', , option_price=10.6769)

In [10]:
test_option.calculate_greeks()

{'spot': 317.30999755859375,
 'strike': 330.0,
 'volatility': 0.27780873565673825,
 'price': 10.676945471721842,
 'delta': 0.4213320007644451,
 'gamma': 0.003952099314689509,
 'vega': 0.5361716371989438}

In [11]:
european_call_option = EuropeanOption(ticker = "AAPL", strike = strike, expiry_date = "2026-09-18", option_type = "call")

In [12]:
european_call_option.calculate_greeks()

{'spot': 317.30999755859375,
 'strike': 330.0,
 'volatility': 0.27780873565673825,
 'price': 10.648076414033937,
 'delta': 0.4159206965615215,
 'gamma': 0.010321254562861953,
 'vega': 0.5309014858208769}

In [12]:
european_call_option = EuropeanOption(ticker = "AAPL", strike = strike, expiry_date = "2026-09-18", option_type = "call")
print(european_call_option)
european_put_option = EuropeanOption(ticker = "AAPL", strike = strike, expiry_date = "2026-09-18", option_type = "put")
print(european_put_option)
american_call_option = AmericanOption(ticker = "AAPL", strike = strike, expiry_date = "2026-09-18", option_type = "call")
print(american_call_option)
american_put_option = AmericanOption(ticker = "AAPL", strike = strike, expiry_date = "2026-09-18", option_type = "put")
print(american_put_option)

EuropeanOption(ticker='AAPL', strike='300.0', expiry_date='2026-09-18', option_type='call', underlying_spot='317.21', volatility='0.3002', risk_free_rate='0.0373', , option_price=27.2350)
EuropeanOption(ticker='AAPL', strike='300.0', expiry_date='2026-09-18', option_type='put', underlying_spot='317.21', volatility='0.2722', risk_free_rate='0.0373', , option_price=94.1695)
AmericanOption(ticker='AAPL', strike='300.0', expiry_date='2026-09-18', option_type='call', underlying_spot='317.21', volatility='0.3002', risk_free_rate='0.0373', , option_price=27.2367)
AmericanOption(ticker='AAPL', strike='300.0', expiry_date='2026-09-18', option_type='put', underlying_spot='317.21', volatility='0.2722', risk_free_rate='0.0373', , option_price=6.7746)


In [7]:
test_option.__repr__()

"EuropeanOption(ticker='AAPL', strike='300.0', expiry_date='2026-09-18', option_type='call', underlying_spot='317.20', volatility='0.3000', risk_free_rate='0.0373', , option_price=27.2137)"

In [ ]:
def calculate_historical_volatility(ticker_symbol, lookback_days=30):
    """
    Retrieves data via yfinance and calculates annualized historical volatility 
    based on logarithmic returns over a specific lookback window.
    """
    # 1. Fetch slightly more historical days to guarantee we get our full trading day window
    # (Since markets close on weekends/holidays, 45 calendar days ~ 30 trading days)
    calendar_window = int(lookback_days * 1.5)
    
    ticker = yf.Ticker(ticker_symbol)
    df = ticker.history(period=f"{calendar_window}d")
    
    # Slice the exact amount of recent trading data we need
    df = df.tail(lookback_days)
    
    # 2. Calculate daily logarithmic returns: ln(Price_t / Price_t-1)
    df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))
    
    # 3. Calculate the standard deviation of daily log returns
    daily_volatility = df['Log_Return'].std()
    
    # 4. Annualize the volatility 
    # (We multiply by the square root of 252 because variance scales with time)
    annualized_volatility = daily_volatility * np.sqrt(252)
    
    return annualized_volatility


In [41]:
get_current_risk_free_rate()

0.037230000495910645

In [20]:
# 1. Instantiate the contract
my_call_option = EuropeanOption(
    ticker="AAPL", 
    strike=100.0, 
    expiry_date="2027-01-13"
)

Microsoft: MSFT
Google: GOOGL
NVidia: NVDA
Tesla: TSLA
Amazon: AMZN

In [23]:
# 2. See all available expiration dates
print("Available Expirations:", ticker.options)

# 3. Fetch the option chain for a specific date (e.g., the first available date)
target_date = ticker.options[0]
opt_chain = ticker.option_chain(target_date)

# 4. Separate into Puts and Calls DataFrames
calls_df = opt_chain.calls
puts_df = opt_chain.puts

# Look at the first few rows of calls
print(calls_df[['contractSymbol', 'strike', 'lastPrice', 'bid', 'ask', 'impliedVolatility']].head())

Available Expirations: ('2026-07-13', '2026-07-15', '2026-07-17', '2026-07-20', '2026-07-22', '2026-07-24', '2026-07-31', '2026-08-07', '2026-08-14', '2026-08-21', '2026-08-28', '2026-09-18', '2026-10-16', '2026-11-20', '2026-12-18', '2027-01-15', '2027-02-19', '2027-03-19', '2027-06-17', '2027-09-17', '2027-12-17', '2028-01-21', '2028-03-17', '2028-12-15')
        contractSymbol  strike  lastPrice  bid  ask  impliedVolatility
0  AAPL260713C00205000   205.0     102.68  0.0  0.0            0.00001
1  AAPL260713C00210000   210.0      97.71  0.0  0.0            0.00001
2  AAPL260713C00215000   215.0      99.91  0.0  0.0            0.00001
3  AAPL260713C00220000   220.0      94.94  0.0  0.0            0.00001
4  AAPL260713C00225000   225.0      58.24  0.0  0.0            0.00001
